In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# SOIL DATASET - PRODUCTION QUALITY CLEANING SCRIPT
# ============================================================

INPUT_FILE = "soil_dataset_india.csv"
OUTPUT_FILE = "soil_dataset_india_clean.csv"

# Create output directory if it doesn't exist
Path("data/processed").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("SOIL DATASET CLEANING")
print("=" * 60)

print("Original shape:", df.shape)

# ------------------------------------------------------------
# 2. Check required columns
# ------------------------------------------------------------

required_columns = [
    "country",
    "latitude",
    "longitude",
    "soil_ph",
    "organic_carbon",
    "sand",
    "silt",
    "clay",
    "nitrogen"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Keep only required columns
df = df[required_columns].copy()

# ------------------------------------------------------------
# 3. Clean country column
# ------------------------------------------------------------

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)

df["country"] = df["country"].replace({
    "IND": "India",
    "INDIA": "India",
    "india": "India"
})

# ------------------------------------------------------------
# 4. Convert numerical columns
# ------------------------------------------------------------

numeric_columns = [
    "latitude",
    "longitude",
    "soil_ph",
    "organic_carbon",
    "sand",
    "silt",
    "clay",
    "nitrogen"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 5. Replace invalid organic carbon values
# ------------------------------------------------------------

# -95 and -98 are treated as missing-value codes,
# not actual organic carbon measurements.

df["organic_carbon"] = df["organic_carbon"].replace(
    [-95, -98],
    pd.NA
)

# ------------------------------------------------------------
# 6. Validate latitude
# ------------------------------------------------------------

invalid_latitude = ~df["latitude"].between(-90, 90)

df.loc[
    invalid_latitude,
    "latitude"
] = pd.NA

# ------------------------------------------------------------
# 7. Validate longitude
# ------------------------------------------------------------

invalid_longitude = ~df["longitude"].between(-180, 180)

df.loc[
    invalid_longitude,
    "longitude"
] = pd.NA

# ------------------------------------------------------------
# 8. Validate soil pH
# ------------------------------------------------------------

# Valid physical pH scale = 0 to 14

invalid_ph = ~df["soil_ph"].between(0, 14)

df.loc[
    invalid_ph,
    "soil_ph"
] = pd.NA

# ------------------------------------------------------------
# 9. Validate soil texture
# ------------------------------------------------------------

# Sand + Silt + Clay should approximately equal 100%.
# A tolerance of ±1 is allowed for rounding differences.

texture_sum = df[
    ["sand", "silt", "clay"]
].sum(
    axis=1,
    min_count=3
)

invalid_texture = (
    texture_sum.notna()
    & (abs(texture_sum - 100) > 1)
)

invalid_texture_count = invalid_texture.sum()

# Do NOT invent values.
# Mark inconsistent texture measurements as missing.

df.loc[
    invalid_texture,
    ["sand", "silt", "clay"]
] = pd.NA

# ------------------------------------------------------------
# 10. Remove negative soil measurements
# ------------------------------------------------------------

non_negative_columns = [
    "organic_carbon",
    "sand",
    "silt",
    "clay",
    "nitrogen"
]

for column in non_negative_columns:

    invalid_negative = df[column] < 0

    df.loc[
        invalid_negative,
        column
    ] = pd.NA

# IMPORTANT:
# nitrogen = 0 is kept because it may be a genuine
# measured value. We do NOT treat zero as missing.

# ------------------------------------------------------------
# 11. Remove exact duplicate rows
# ------------------------------------------------------------

rows_before_duplicates = len(df)

df = df.drop_duplicates()

df = df.reset_index(drop=True)

duplicates_removed = (
    rows_before_duplicates - len(df)
)

# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(
    "Rows before cleaning:",
    rows_before_duplicates
)

print(
    "Rows after cleaning:",
    len(df)
)

print(
    "Duplicate rows removed:",
    duplicates_removed
)

print(
    "Invalid soil texture rows:",
    invalid_texture_count
)

print("\nMissing values after cleaning:")

print(
    df.isnull().sum()
)

print("\nData types:")

print(
    df.dtypes
)

# ------------------------------------------------------------
# 13. Save cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("CLEANING COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "Cleaned file saved to:",
    OUTPUT_FILE
)

SOIL DATASET CLEANING
Original shape: (1093, 9)

CLEANING SUMMARY
Rows before cleaning: 1093
Rows after cleaning: 1093
Duplicate rows removed: 0
Invalid soil texture rows: 0

Missing values after cleaning:
country            0
latitude           0
longitude          0
soil_ph           17
organic_carbon    17
sand              17
silt              17
clay              17
nitrogen           0
dtype: int64

Data types:
country           string[python]
latitude                 float64
longitude                float64
soil_ph                  float64
organic_carbon            object
sand                     float64
silt                     float64
clay                     float64
nitrogen                 float64
dtype: object

CLEANING COMPLETED SUCCESSFULLY
Cleaned file saved to: soil_dataset_india_clean.csv
